In [2]:
import pandas as pd
import tensorflow as tf
import sklearn
import scikeras

In [3]:
pd.__version__, tf.__version__, sklearn.__version__, scikeras.__version__

('3.1.1', '2.21.0', '1.9.1', '0.13.0')

In [4]:
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import cross_val_score
from tensorflow.keras.models import Sequential
from tensorflow.keras import backend as k

In [5]:
from ucimlrepo import fetch_ucirepo
import matplotlib.pyplot as plt

## Data Loading

In [6]:
# fetch dataset 
breast_cancer_wisconsin_diagnostic = fetch_ucirepo(id=17) 

X = breast_cancer_wisconsin_diagnostic.data.features
y = breast_cancer_wisconsin_diagnostic.data.targets

# M = 1, B = 0
y = y["Diagnosis"].map({"B": 0, "M": 1}).to_frame()

In [7]:
# features
X.head()

,radius1,texture1,perimeter1,area1,smoothness1,compactness1,concavity1,concave_points1,symmetry1,fractal_dimension1,...,radius3,texture3,perimeter3,area3,smoothness3,compactness3,concavity3,concave_points3,symmetry3,fractal_dimension3
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [8]:
# targets
y.head()

,Diagnosis
0,1
1,1
2,1
3,1
4,1


## Neural Networ Structure Creating

In [9]:
def create_network():
    k.clear_session()
    neural_network = Sequential([
        tf.keras.layers.InputLayer(shape=(30,)),
        tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
        tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
        tf.keras.layers.Dense(units=1, activation='sigmoid')    
    ])
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001, clipvalue=0.5)
    neural_network.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['binary_accuracy'])
    return neural_network

In [10]:
neural_network = KerasClassifier(model=create_network, epochs=100, batch_size=10)

In [11]:
results = cross_val_score(estimator=neural_network, X=X, y=y, cv=10, scoring='accuracy')


Epoch 1/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - binary_accuracy: 0.6289 - loss: 0.6620
Epoch 2/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.8652 - loss: 0.5123
Epoch 3/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.8965 - loss: 0.3755
Epoch 4/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.9004 - loss: 0.3065
Epoch 5/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - binary_accuracy: 0.9219 - loss: 0.2362
Epoch 6/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.9180 - loss: 0.2615
Epoch 7/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.9043 - loss: 0.2699
Epoch 8/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.9062 - loss: 0.2396
Epoch 9/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.9180 - loss: 0.2470
Epoch 10/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.9199 - loss: 0.2315
Epoch 11/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.9219 

In [12]:
results, results.mean(), results.std()

(array([0.9122807 , 0.9122807 , 0.92982456, 0.77192982, 0.8245614 ,
        0.96491228, 0.89473684, 0.98245614, 0.9122807 , 0.85714286]),
 np.float64(0.8962406015037594),
 np.float64(0.06014202114073441))

### z-score

In [13]:
import numpy as np
import numpy.typing as npt

def z_score(results: npt.NDArray[np.float64]) -> npt.NDArray[np.float64]:
    mean = results.mean()
    std = results.std()
    return (results - mean) / std

In [14]:
z_score(results)

array([ 0.26670371,  0.26670371,  0.5584109 , -2.06695376, -1.19183221,
        1.14182526, -0.02500347,  1.43353245,  0.26670371, -0.6500903 ])